# Comparaison d'algos en régression

Nous allons comparer les différentes méthodes de régression que nous avons vues jusqu'à présent, les moindres carrés avec toutes les variables, puis les moindres carrés avec les variables sélectionnées par le critère BIC avec un algo backward et par AIC.
Le nombre de bloc de la validation croisée vaut $k=10$ mais peut être modifié par l'utilisateur. Les données s'appellent *don* et la variable d'intérêt $Y$

In [1]:
!pip install catboost 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 47.5 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]1/2 [catboost]


In [ ]:
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from patsy import dmatrix
#import ols_step_sk
###
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.linear_model import RidgeCV, ElasticNetCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
###
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error

In [3]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

In [4]:
don = pd.read_csv("https://regression-avec-python.github.io/donnees/ozone.txt", header=0, sep=";",index_col=0)
don.rename(columns={"O3":"Y"},inplace=True)

Codage des variables qualitatives nebu et vent

In [5]:
don = pd.read_csv("https://regression-avec-python.github.io/donnees/ozone_transf.txt", header = 0, sep = ";", index_col=0)
print(don.shape)
don.rename(columns={"maxO3":"Y"},inplace=True)

(1366, 22)


In [6]:
nomsvar = list(don.columns.difference(["Y"]))
print(nomsvar)
#design matrix
formule = "~" + "+".join(nomsvar)
print(formule)
dsX = dmatrix(formule,don)
X = np.asarray(dsX)[:,1:]
Y = don["Y"].to_numpy()

['Ne12', 'Ne15', 'Ne18', 'Ne6', 'Ne9', 'T12', 'T15', 'T18', 'T6', 'T9', 'Vx12', 'Vx15', 'Vx18', 'Vx6', 'Vx9', 'Vy12', 'Vy15', 'Vy18', 'Vy6', 'Vy9', 'maxO3v']
~Ne12+Ne15+Ne18+Ne6+Ne9+T12+T15+T18+T6+T9+Vx12+Vx15+Vx18+Vx6+Vx9+Vy12+Vy15+Vy18+Vy6+Vy9+maxO3v


In [7]:
nb=10 ###nb de blocs
tmp = np.arange(don.shape[0])%nb
rng = np.random.default_rng(seed=1234)
bloc = rng.choice(tmp,size=don.shape[0],replace=False) ### l'indice d'appartenance aux blocs

PREV = pd.DataFrame({"bloc":bloc,"Y":don["Y"],"MCO":0.0,"BIC":0.0,"AIC":0.0,
                    "lasso":0.0,"ridge":0.0,"elastic":0.0,"ridgeGS":0.0,'arbre':0.0,'foret':0.0,
                    "gbm1":0.0,"gbm2":0.0,"gbms1":0.0,"gbms2":0.0,"cat":0.0})

In [8]:
cr = StandardScaler()
kf = KFold(n_splits=10, shuffle=True, random_state=0)
lassocv = LassoCV(cv=kf)
pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
etape_lassocv = pipe_lassocv.named_steps["lassocv"]
lambdaoptlasso=[]
nopt1=[]
nopt2=[]
enetcv = ElasticNetCV(cv=kf,max_iter=10000)
pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
ridge = Ridge()
pipe_ridge = Pipeline(steps=[("cr", cr), ("ridge", ridge)])

In [ ]:
for i in np.arange(nb):
    print(i)
    Xapp = X[bloc!=i,:]
    Xtest = X[bloc==i,:]
    Yapp = don[bloc!=i]["Y"]
    Ytest = don[bloc==i]["Y"]
    #### reg
    reg = LinearRegression()
    reg.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"MCO"] = reg.predict(Xtest)
    ### bic
    #inst_reg_bic = ols_step_sk.LinearRegressionSelectionFeatureIC(verbose=1,crit="bic")
    #reg_bic = inst_reg_bic.fit(X=Xapp, y=Yapp)
    #PREV.loc[PREV.bloc==i,"BIC"] = reg_bic.predict(Xtest)
    ### aic
    #inst_reg_aic = ols_step_sk.LinearRegressionSelectionFeatureIC(verbose=1,crit="aic")
    #reg_aic = inst_reg_aic.fit(X=Xapp, y=Yapp)
    #PREV.loc[PREV.bloc==i,"AIC"] = reg_aic.predict(Xtest)
    ###lasso
    pipe_lassocv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"lasso"] = pipe_lassocv.predict(Xtest)
    ###elastic net pondération 1/2
    pipe_enetcv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"elastic"] = pipe_enetcv.predict(Xtest)
    ###ridge avec le chemin de reg qui vaut 100 le chemin de lasso
    alphasridge = 100*etape_lassocv.alphas_
    ridgecv = RidgeCV(cv=kf,alphas=alphasridge)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"ridge"] = pipe_ridgecv.predict(Xtest)
    ## ridge grid searchparams lambda
    path_ridge = alphasridge   
    param_grid_ridge = {"ridge__alpha": path_ridge}
    cv_ridge = GridSearchCV(pipe_ridge, param_grid_ridge, cv=kf, scoring = "neg_mean_squared_error", n_jobs=3).fit(Xapp, Yapp)
    PREV.loc[PREV.bloc==i,"ridgeGS"] = cv_ridge.predict(Xtest)
    ###arbre
    arbre = DecisionTreeRegressor(min_samples_leaf=5).fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"arbre"] = arbre.predict(Xtest)
    ###foret
    foret = RandomForestRegressor().fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"foret"] = foret.predict(Xtest)
    ###gradient boosting
    params = dict(n_estimators=1000, max_depth=1, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingRegressor(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PREV.loc[PREV.bloc==i,"gbm1"] = gbm_early_stopping.predict(Xtest)
    params = dict(n_estimators=1000, max_depth=2, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingRegressor(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PREV.loc[PREV.bloc==i,"gbm2"] = gbm_early_stopping.predict(Xtest)
    ### stochastic gradient boosting
    params = dict(n_estimators=5000, max_depth=1, learning_rate=0.1, random_state=42,subsample=0.2)
    gbm_early_stopping = GradientBoostingRegressor(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PREV.loc[PREV.bloc==i,"gbms1"] = gbm_early_stopping.predict(Xtest)
    params = dict(n_estimators=5000, max_depth=2, learning_rate=0.1, random_state=42, subsample=0.2)
    gbm_early_stopping = GradientBoostingRegressor(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PREV.loc[PREV.bloc==i,"gbms2"] = gbm_early_stopping.predict(Xtest)
    ###Catboost
    X_train, X_valid, y_train, y_valid = train_test_split(Xapp,Yapp,test_size=0.1,random_state=42)
    model = CatBoostRegressor(iterations=5000,learning_rate=0.1,subsample=0.1,loss_function="RMSE",
    random_seed=42)
    model.fit(X_train,y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=10)
    print(model.get_best_iteration())
    PREV.loc[PREV.bloc==i,"cat"] = model.predict(Xtest)

0
124
62
109
62
0:	learn: 21.7968339	test: 23.0002164	best: 23.0002164 (0)	total: 47.2ms	remaining: 3m 55s
1:	learn: 20.6830028	test: 21.7919403	best: 21.7919403 (1)	total: 48.3ms	remaining: 2m
2:	learn: 19.8734083	test: 20.9804081	best: 20.9804081 (2)	total: 49.4ms	remaining: 1m 22s
3:	learn: 19.0239193	test: 20.0991928	best: 20.0991928 (3)	total: 50.5ms	remaining: 1m 3s
4:	learn: 18.2427125	test: 19.2912219	best: 19.2912219 (4)	total: 51.5ms	remaining: 51.5s
5:	learn: 17.5534135	test: 18.5456166	best: 18.5456166 (5)	total: 52.4ms	remaining: 43.6s
6:	learn: 16.9165687	test: 17.8763429	best: 17.8763429 (6)	total: 53.5ms	remaining: 38.2s
7:	learn: 16.3603134	test: 17.2927991	best: 17.2927991 (7)	total: 54.5ms	remaining: 34s
8:	learn: 15.8362752	test: 16.6771220	best: 16.6771220 (8)	total: 55.5ms	remaining: 30.8s
9:	learn: 15.4064300	test: 16.2158307	best: 16.2158307 (9)	total: 56.4ms	remaining: 28.1s
10:	learn: 14.9632795	test: 15.7412940	best: 15.7412940 (10)	total: 57.3ms	remaining: 2

On a tout préparé, on peut envoyer

In [10]:
prev = PREV.iloc[:,1:]
np.round((prev.sub(PREV.Y, axis=0)**2).mean(),2)

Y             0.00
MCO         187.51
BIC        7745.43
AIC        7745.43
lasso       187.05
ridge       187.45
elastic     187.20
ridgeGS     187.43
arbre       249.36
foret       155.08
gbm1        160.72
gbm2        156.31
gbms1       164.92
gbms2       156.31
cat         154.87
dtype: float64